# How this notebook is laid out - Building an MDP Agent

1) First load required packages
2) MDB Setup
3) MDP Experiments (Parts 3-5)

No imports of MDP_Agent.py because this is a fully contained notebook.

In [1]:
# Verify required packages are available
import sys
print(f"Python version: {sys.version}")

try:
    import numpy as np
    print(f"✓ numpy {np.__version__} is available")
except ImportError as e:
    print(f"✗ numpy not available: {e}")

try:
    import matplotlib.pyplot as plt
    print("✓ matplotlib is available")
except ImportError as e:
    print(f"✗ matplotlib not available: {e}")

print("All required packages are ready!")

Python version: 3.12.13 (main, Mar  3 2026, 12:39:30) [Clang 17.0.0 (clang-1700.6.3.2)]
✓ numpy 2.4.1 is available
✓ matplotlib is available
All required packages are ready!


This solution builds on the MDP code from Section 4.4, adapted for the 3×4 grid from Problem 4.3 (with a wall at ). We add an environment simulator, an agent loop, and experiments.

## Parts 1–2: Environment Simulator and Agent Loop
# MDP Agent: Environment Simulator and Agent Loop
This script builds on the MDP code from Section 4.4. We adapt the 4×4 warehouse MDP to the 3×4 grid from Problem 4.3 (with a wall at ), then add an environment simulator and an agent loop.

## MDP Setup (3×4 Grid)
We redefine the MDP for the 3×4 grid with a wall at (2,2), goal at (4,3), and hazard at (4,2).

In [3]:
import random
from collections import Counter
# Grid dimensions
WIDTH, HEIGHT = 4, 3
# Terminal states and their rewards
GOAL = (4, 3)
HAZARD = (4, 2)
TERMINALS = {GOAL: +1.0, HAZARD: -1.0}
# Living reward for non-terminal states
LIVING_REWARD = -0.04
# Wall position
WALL = (2, 2)
# All valid states: every grid cell except the wall
STATES = [
    (x, y)
    for x in range(1, WIDTH + 1)
    for y in range(1, HEIGHT + 1)
    if (x, y) != WALL
]
# Actions and their (dx, dy) displacements
ACTIONS = {
    "North": (0, 1),
    "South": (0, -1),
    "East": (1, 0),
    "West": (-1, 0),
}
ARROWS = {"North": "\u2191", "South": "\u2193", "East": "\u2192", "West": "\u2190"}
def reward(state):
    """R(s): immediate reward for being in state s."""
    if state in TERMINALS:
        return TERMINALS[state]
    return LIVING_REWARD
def get_perpendicular(action):
    """Return the two actions perpendicular to the given action."""
    if action in ("North", "South"):
        return ["West", "East"]
    else:
        return ["North", "South"]
def attempt_move(state, action):
    """Return the state that results from attempting to move in the
    given direction. If the move would leave the grid or hit a wall,
    return the original state."""
    dx, dy = ACTIONS[action]
    nx, ny = state[0] + dx, state[1] + dy
    if 1 <= nx <= WIDTH and 1 <= ny <= HEIGHT and (nx, ny) != WALL:
        return (nx, ny)
    return state
def transitions(state, action):
    """T(s' | s, a): return a dict {s': probability}."""
    if state in TERMINALS:
        return {}
    outcomes = {}
    intended = attempt_move(state, action)
    outcomes[intended] = outcomes.get(intended, 0) + 0.8
    for perp in get_perpendicular(action):
        drifted = attempt_move(state, perp)
        outcomes[drifted] = outcomes.get(drifted, 0) + 0.1
    return outcomes
def value_iteration(gamma=0.99, epsilon=1e-6):
    """Run value iteration and return the converged value function
    and the number of iterations."""
    V = {s: 0.0 for s in STATES}
    iteration = 0
    while True:
        V_new = {}
        delta = 0
        for s in STATES:
            if s in TERMINALS:
                V_new[s] = reward(s)
                continue
            best_value = float("-inf")
            for a in ACTIONS:
                expected = sum(
                    prob * V[s_next]
                    for s_next, prob in transitions(s, a).items()
                )
                best_value = max(best_value, expected)
            V_new[s] = reward(s) + gamma * best_value
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        iteration += 1
        if delta < epsilon:
            break
    return V, iteration
def extract_policy(V, gamma=0.99):
    """Compute the optimal policy from a converged value function."""
    policy = {}
    for s in STATES:
        if s in TERMINALS:
            policy[s] = None
            continue
        best_action = None
        best_value = float("-inf")
        for a in ACTIONS:
            expected = sum(
                prob * V[s_next]
                for s_next, prob in transitions(s, a).items()
            )
            value = reward(s) + gamma * expected
            if value > best_value:
                best_value = value
                best_action = a
        policy[s] = best_action
    return policy

# Part 1: Environment Simulator
The simulate_step function samples a next state from the transition distribution T(s' | s,a) using random.choices.

In [12]:
def simulate_step(state, action):
    """Sample a next state from T(s' | s, a)."""
    dist = transitions(state, action)
    states = list(dist.keys())
    probs = list(dist.values())
    return random.choices(states, weights=probs, k=1)[0]

## Verification
Call simulate_step((3, 1), "North") 10,000 times and check that the empirical frequencies approximate 80/10/10.

In [4]:
random.seed(42)
counts = Counter()
for _ in range(10_000):
    counts[simulate_step((3, 1), "North")] += 1
print("Empirical transition frequencies from (3,1), action North:")
for s, c in sorted(counts.items()):
    print(f"  {s}: {c/10_000:.3f}")

Empirical transition frequencies from (3,1), action North:
  (2, 1): 0.099
  (3, 2): 0.803
  (4, 1): 0.098


## Part 2: Agent Loop
The run_episode function simulates a full episode: the agent follows the policy, accumulates reward, and stops at a terminal state or after max_steps.

In [9]:
def run_episode(policy, start=(1, 1), max_steps=100):
    """Simulate one episode following the given policy.
    Returns:
        trajectory: list of states visited (including start)
        total_reward: sum of R(s) over all visited states
        outcome: "goal", "hazard", or "timeout"
    """
    state = start
    trajectory = [state]
    total_reward = reward(state)
    for step in range(max_steps):
        if state in TERMINALS:
            break
        action = policy[state]
        state = simulate_step(state, action)
        trajectory.append(state)
        total_reward += reward(state)
    if state == GOAL:
        outcome = "goal"
    elif state == HAZARD:
        outcome = "hazard"
    else:
        outcome = "timeout"
    return trajectory, total_reward, outcome

Running 1000 Episodes with the Optimal Policy

In [6]:
random.seed(42)
V, num_iters = value_iteration(gamma=0.99)
optimal_policy = extract_policy(V, gamma=0.99)
print(f"Value iteration converged in {num_iters} iterations.\n")
results = [run_episode(optimal_policy) for _ in range(1000)]
outcomes = [r[2] for r in results]
rewards = [r[1] for r in results]
goal_rate = outcomes.count("goal") / 1000
hazard_rate = outcomes.count("hazard") / 1000
avg_reward = sum(rewards) / 1000
print(f"Goal reached:  {goal_rate:.3f}")
print(f"Hazard hit:    {hazard_rate:.3f}")
print(f"Avg reward:    {avg_reward:.3f}")

Value iteration converged in 27 iterations.

Goal reached:  0.982
Hazard hit:    0.018
Avg reward:    0.697


The optimal policy reaches the goal about 98% of the time, hits the hazard about 2% of the time (due to unlucky drift sequences), and never times out. The average total reward is positive, reflecting that the goal’s +1 reward outweighs the accumulated living penalties and occasional hazard hits.

## Parts 3–5: Greedy Comparison, Discount Factor, and Harder Warehouse
# MDP Experiments: Greedy Comparison, Discount Factor, and Harder Warehouse
This script runs the experiments from Parts 3–5 of the MDP Agent exercise, using the simulator and agent loop from mdp_agent (above)

In [10]:
import random
from collections import Counter

# Build an in-notebook mdp_agent bundle from previously defined symbols.
required_symbols = ["GOAL", "HAZARD", "STATES", "TERMINALS"]
optional_symbols = [
    "ACTIONS", "ARROWS", "HEIGHT", "WIDTH", "WALL",
    "extract_policy", "reward", "run_episode", "value_iteration",
]

missing = [name for name in required_symbols if name not in globals()]
if missing:
    raise NameError(
        "Missing MDP setup symbols: " + ", ".join(missing) + ". Run the MDP setup cell first."
    )

mdp_agent = {name: globals()[name] for name in required_symbols}
mdp_agent.update({name: globals()[name] for name in optional_symbols if name in globals()})
print("mdp_agent ready (using definitions from earlier notebook cells).")

mdp_agent ready (using definitions from earlier notebook cells).


# Part 3: Comparison with a Naive Greedy Policy
The greedy policy always moves in the direction of the goal, ignoring the hazard entirely.

In [7]:
def greedy_policy_action(state):
    """Move toward the goal, ignoring hazards."""
    gx, gy = mdp_agent["GOAL"]
    sx, sy = state
    if sx < gx:
        return "East"
    elif sx > gx:
        return "West"
    elif sy < gy:
        return "North"
    else:
        return "South"

greedy_policy = {
    s: greedy_policy_action(s)
    for s in mdp_agent["STATES"]
    if s not in mdp_agent["TERMINALS"]
}
greedy_policy[mdp_agent["GOAL"]] = None
greedy_policy[mdp_agent["HAZARD"]] = None

Running 1000 Episodes with Each Policy

In [13]:
random.seed(42)

value_iteration_fn = mdp_agent.get("value_iteration", globals().get("value_iteration"))
extract_policy_fn = mdp_agent.get("extract_policy", globals().get("extract_policy"))
run_episode_fn = mdp_agent.get("run_episode", globals().get("run_episode"))

missing = [
    name
    for name, fn in [
        ("value_iteration", value_iteration_fn),
        ("extract_policy", extract_policy_fn),
        ("run_episode", run_episode_fn),
    ]
    if fn is None
]
if missing:
    raise NameError(
        "Missing required functions: " + ", ".join(missing) +
        ". Run the Part 2 Agent Loop cell before this experiment."
    )

V, _ = value_iteration_fn(gamma=0.99)
optimal_policy = extract_policy_fn(V, gamma=0.99)
opt_results = [run_episode_fn(optimal_policy) for _ in range(1000)]
opt_outcomes = [r[2] for r in opt_results]
opt_rewards = [r[1] for r in opt_results]
greedy_results = [run_episode_fn(greedy_policy) for _ in range(1000)]
greedy_outcomes = [r[2] for r in greedy_results]
greedy_rewards = [r[1] for r in greedy_results]
print("Optimal policy:")
print(f"  Goal reached:  {opt_outcomes.count('goal') / 1000:.3f}")
print(f"  Hazard hit:    {opt_outcomes.count('hazard') / 1000:.3f}")
print(f"  Avg reward:    {sum(opt_rewards) / 1000:.3f}")
print("\nGreedy policy:")
print(f"  Goal reached:  {greedy_outcomes.count('goal') / 1000:.3f}")
print(f"  Hazard hit:    {greedy_outcomes.count('hazard') / 1000:.3f}")
print(f"  Avg reward:    {sum(greedy_rewards) / 1000:.3f}")

Optimal policy:
  Goal reached:  0.982
  Hazard hit:    0.018
  Avg reward:    0.697

Greedy policy:
  Goal reached:  0.053
  Hazard hit:    0.947
  Avg reward:    -1.114


The greedy policy is catastrophic on this grid: it reaches the goal only ~5% of the time and hits the hazard ~95% of the time. The reason is structural: the greedy policy sends the robot East along the bottom row to (4,1), then North—directly into the hazard at (4,2) with 80% probability. The optimal policy avoids the hazard column entirely by routing up the left side and across the top row, reaching the goal ~98% of the time.

Key insight: The MDP-optimal policy sacrifices short-term efficiency (taking longer paths) for dramatically better outcomes. The greedy policy optimizes for distance to the goal but ignores both the stochastic transition model and the hazard.

# Part 4: Discount Factor Experiment
We compute optimal policies for several values of  and compare their performance.

In [14]:
random.seed(42)
print("Discount factor experiment (living reward = -0.04):\n")
for gamma in [0.1, 0.5, 0.9, 0.99]:
    V, _ = value_iteration(gamma=gamma)
    policy = extract_policy(V, gamma=gamma)
    results = [run_episode(policy) for _ in range(1000)]
    outcomes = [r[2] for r in results]
    rewards = [r[1] for r in results]
    goal_rate = outcomes.count("goal") / 1000
    hazard_rate = outcomes.count("hazard") / 1000
    avg_reward = sum(rewards) / 1000
    print(f"gamma={gamma:.2f}  goal={goal_rate:.3f}  "
          f"hazard={hazard_rate:.3f}  avg_reward={avg_reward:.3f}")

Discount factor experiment (living reward = -0.04):

gamma=0.10  goal=1.000  hazard=0.000  avg_reward=0.614
gamma=0.50  goal=0.965  hazard=0.035  avg_reward=0.670
gamma=0.90  goal=0.978  hazard=0.022  avg_reward=0.700
gamma=0.99  goal=0.984  hazard=0.016  avg_reward=0.704


Analysis: On this compact 3×4 grid, the hazard’s immediate reward of -1 is severe enough that even a myopic agent avoids it. All four discount factors produce policies that reach the goal at high rates (97–100%). The variation is modest because the grid is small: the optimal path is only a few steps long regardless of how far ahead the agent plans.

gamma = 0.1 : Despite extreme myopia, the agent avoids the hazard because its -1 reward dominates the discounted future at any horizon. The 100% goal rate in this run reflects the short path and favorable random seed.
gamma = 0.5 : Slightly lower goal rate (~97%) as the agent weighs the living penalty more against longer detours.
gamma = 0.9 and gamma = 0.99: Nearly identical performance (~98% goal rate). With enough foresight the policy stabilizes.
On a larger grid (like the 4×4 warehouse in Section 4.4), the discount factor has a more dramatic effect: a myopic agent cannot “see” distant hazards and takes riskier shortcuts.

# Part 5: A Harder Warehouse
We add a second hazard at  with reward , recompute the optimal policy, and evaluate it.

In [15]:
random.seed(42)
TERMINALS[(2, 3)] = -1.0
V, _ = value_iteration(gamma=0.99)
policy = extract_policy(V, gamma=0.99)
print("Policy with second hazard at (2,3):\n")
for y in range(HEIGHT, 0, -1):
    row = []
    for x in range(1, WIDTH + 1):
        s = (x, y)
        if s == GOAL:
            row.append(" GOAL")
        elif s in TERMINALS and TERMINALS[s] < 0:
            row.append(" HAZD")
        elif s not in STATES:
            row.append(" WALL")
        else:
            row.append(f"  {ARROWS[policy[s]]} ")
    print("".join(row))
results = [run_episode(policy) for _ in range(1000)]
outcomes = [r[2] for r in results]
rewards = [r[1] for r in results]
print(f"\nGoal reached:  {outcomes.count('goal') / 1000:.3f}")
print(f"Hazard hit:    {outcomes.count('hazard') / 1000:.3f}")
print(f"Avg reward:    {sum(rewards) / 1000:.3f}")

Policy with second hazard at (2,3):

  ↓  HAZD  →  GOAL
  ↓  WALL  ↑  HAZD
  →   →   ↑   ← 

Goal reached:  0.856
Hazard hit:    0.144
Avg reward:    0.458


Analysis: The second hazard at (2,3) blocks the upper-left corridor that the original policy used. The robot can no longer go East from (1,3) since that enters the new hazard with 80% probability. The bottom-left states now point East along the bottom row, feeding into the x = 3 column which routes North to (3,3) and then East to the goal. States (1,2) and (1,3) point South to escape the now-dead-end left column. The goal-reach rate drops from ~98% to ~86% and the hazard-hit rate increases from ~2% to ~14% because there are now two hazards to accidentally drift into and the safe corridor is narrower.